## 1. Hugging Face Login

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        if self.d_head % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")

        inv_freq = 1.0 / (
            10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        )
        angles = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("rope_cos", angles.cos(), persistent=False)
        self.register_buffer("rope_sin", angles.sin(), persistent=False)
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def apply_rope(self, q, k, offset=0):
        T = q.size(2)
        cos = self.rope_cos[offset:offset + T][None, None].to(q.dtype)
        sin = self.rope_sin[offset:offset + T][None, None].to(q.dtype)

        def rotate(x):
            first, second = x.chunk(2, dim=-1)
            return torch.cat(
                [first * cos - second * sin, first * sin + second * cos],
                dim=-1,
            )

        return rotate(q), rotate(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k = self.apply_rope(q, k)

        scores = q @ k.transpose(-2, -1) * self.attention_scale

        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        weighted_sum = attn_weights @ v
        weighted_sum = weighted_sum.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(weighted_sum)
        out = self.resid_dropout(out)

        return out

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "rope"   
    attention_scale: float = None      
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        if cfg.pos_emb_type != "rope":
            raise ValueError("This notebook requires a RoPE model/checkpoint.")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape

        x = self.tok_emb(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


# **Phase 7**

In [6]:
import json
import random
import shutil
import zipfile

from pathlib import Path
from dataclasses import asdict
from types import MethodType
from functools import lru_cache

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint as activation_checkpoint

from sklearn.feature_extraction.text import TfidfVectorizer
from huggingface_hub import hf_hub_download


# Folder containing your Phase 1 Model 1 files.
MODEL_DIR = Path(
    '/kaggle/input/datasets/sruthigms/model1-for-phase7-files'
)

OUTPUT_DIR = Path('/kaggle/working/phase7_model1_partial')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

MAX_LENGTH = 1024
CHUNK_OVERLAP = 64
MAX_CHUNKS = 8
CHUNK_BATCH_SIZE = 8

BATCH_SIZE = 8
INDEX_BATCH_SIZE = 8

UNFREEZE_LAST_N = 2
PROJECTION_DIM = 256

HARD_NEGATIVES = 2
RANDOM_NEGATIVES = 1
TEMPERATURE = 0.07

EPOCHS = 7
LEARNING_RATE = 2e-5
PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

use_amp = device.type == 'cuda'

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)


def amp_context():
    return torch.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=use_amp,
    )


def load_json(path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)


print('Device:', device)

Device: cuda


In [7]:
paths = {}

for split, name in [
    ('Train', 'train'),
    ('Dev', 'dev'),
    ('Test', 'test'),
]:
    for filename in [
        f'{name}.json',
        f'{name}_candidates.json',
    ]:
        paths[filename] = Path(
            hf_hub_download(
                repo_id='Exploration-Lab/CS-781-Capstone',
                filename=f'Phase_7_PCR/{split}/{filename}',
                repo_type='dataset',
                token=HF_TOKEN,
            )
        )

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

In [8]:
def read_cases(path, labeled=False):
    data = load_json(path)

    # Support either an ID-keyed dictionary or a list of records.
    if isinstance(data, dict):
        items = data.items()

    elif isinstance(data, list):
        items = []

        for row in data:
            case_id = row.get(
                'id',
                row.get('query_id', row.get('case_id')),
            )
            items.append((case_id, row))

    else:
        raise ValueError(f'Unexpected JSON structure: {path}')

    records = {}

    for case_id, row in items:
        if case_id is None:
            raise ValueError(f'A record has no case ID: {path}')

        case_id = str(case_id)

        if not isinstance(row, dict):
            row = {'text': row}

        text = row['text']

        if isinstance(text, list):
            text = ' '.join(
                sentence
                if isinstance(sentence, str)
                else sentence['text']
                for sentence in text
            )

        if not isinstance(text, str) or not text.strip():
            raise ValueError(
                f'Empty or unsupported text for {case_id}'
            )

        if case_id in records:
            raise ValueError(f'Duplicate ID: {case_id}')

        records[case_id] = {'text': text}

        if labeled:
            positives = row['relevant_candidates']

            if not isinstance(positives, list):
                raise ValueError(
                    'relevant_candidates must be a list of case IDs.'
                )

            records[case_id]['relevant_candidates'] = {
                str(candidate_id) for candidate_id in positives
            }

    if not records:
        raise ValueError(f'No records found: {path}')

    return records


train_queries = read_cases(
    paths['train.json'],
    labeled=True,
)
train_candidates = read_cases(
    paths['train_candidates.json']
)

dev_queries = read_cases(
    paths['dev.json'],
    labeled=True,
)
dev_candidates = read_cases(
    paths['dev_candidates.json']
)

test_queries = read_cases(paths['test.json'])
test_candidates = read_cases(paths['test_candidates.json'])

# Each labeled split must use its corresponding candidate pool.
for queries, candidates in [
    (train_queries, train_candidates),
    (dev_queries, dev_candidates),
]:
    candidate_ids = set(candidates)

    for query_id, row in queries.items():
        missing = row['relevant_candidates'] - candidate_ids

        if missing:
            raise ValueError(
                f'{query_id}: gold IDs are missing from '
                'this split candidate pool.'
            )

print(
    'Train:',
    len(train_queries), 'queries,',
    len(train_candidates), 'candidates',
)
print(
    'Dev:',
    len(dev_queries), 'queries,',
    len(dev_candidates), 'candidates',
)
print(
    'Test:',
    len(test_queries), 'queries,',
    len(test_candidates), 'candidates',
)

Train: 827 queries, 4320 candidates
Dev: 118 queries, 1023 candidates
Test: 237 queries, 1727 candidates


In [9]:
def load_model1(model_dir):
    saved = torch.load(
        model_dir / 'model.pt',
        map_location='cpu',
        weights_only=True,
    )

    config = LMConfig(**saved['model_config'])

    model = DecoderLM(config)
    model.load_state_dict(
        saved['model_state_dict'],
        strict=True,
    )

    state = load_json(model_dir / 'tokenizer.json')

    tok = BytelevelBPE(state['vocab_size'])
    tok.specials = state['specials']
    tok.vocab = state['vocab']
    tok.vocab_size = len(tok.vocab)

    tok.stoi = {
        symbol: i for i, symbol in enumerate(tok.vocab)
    }

    tok.itos = {
        i: symbol for symbol, i in tok.stoi.items()
    }

    tok.merges = [
        tuple(pair) for pair in state['merges']
    ]

    tok.merge_ranks = {
        tuple(row['pair']): (
            row['rank'],
            row['merged_symbol'],
        )
        for row in state['merge_ranks']
    }

    tok.symbol_to_bytes = {
        symbol: bytes(value)
        for symbol, value in state['symbol_to_bytes'].items()
    }

    # Reuse tokenization results for repeated words.
    tok.apply_bpe = lru_cache(maxsize=100000)(tok.apply_bpe)

    if tok.vocab_size != config.vocab_size:
        raise ValueError(
            'Tokenizer and model vocabulary sizes differ.'
        )

    return model, tok, config


phase1_model, tokenizer, cfg = load_model1(MODEL_DIR)

MAX_LENGTH = min(MAX_LENGTH, cfg.max_seq_len)

PAD_ID = tokenizer.stoi['<pad>']
BOS_ID = tokenizer.stoi['<s>']
EOS_ID = tokenizer.stoi['</s>']

assert 0 <= CHUNK_OVERLAP < MAX_LENGTH - 2
assert 1 <= UNFREEZE_LAST_N < cfg.n_layers

print('Maximum chunk length:', MAX_LENGTH)
print('Decoder layers:', cfg.n_layers)

Maximum chunk length: 1024
Decoder layers: 12


In [10]:
def document_chunks(text):
    tokens = tokenizer.encode(text)

    # Reserve positions for BOS and EOS.
    capacity = MAX_LENGTH - 2
    step = capacity - CHUNK_OVERLAP

    chunks = []

    for start in range(0, len(tokens), step):
        chunks.append(tokens[start:start + capacity])

        if start + capacity >= len(tokens):
            break

    if not chunks:
        chunks = [[]]

    # Cover different parts of long documents.
    if len(chunks) > MAX_CHUNKS:
        positions = np.linspace(
            0,
            len(chunks) - 1,
            MAX_CHUNKS,
            dtype=int,
        )
        chunks = [chunks[i] for i in positions]

    return [
        [BOS_ID] + chunk + [EOS_ID]
        for chunk in chunks
    ]


def prepare_cases(records):
    return {
        case_id: document_chunks(row['text'])
        for case_id, row in records.items()
    }


train_query_chunks = prepare_cases(train_queries)
train_candidate_chunks = prepare_cases(train_candidates)

dev_query_chunks = prepare_cases(dev_queries)
dev_candidate_chunks = prepare_cases(dev_candidates)

print('Document chunks prepared.')

Document chunks prepared.


In [11]:
def pcr_attention_forward(self, x):
    batch, length, width = x.shape

    q = self.q_proj(x)
    k = self.k_proj(x)
    v = self.v_proj(x)

    q = q.view(
        batch, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    k = k.view(
        batch, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    v = v.view(
        batch, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    q, k = self.apply_rope(q, k)

    output = F.scaled_dot_product_attention(
        q,
        k,
        v,
        is_causal=True,
        dropout_p=(
            self.attn_dropout.p if self.training else 0.0
        ),
        scale=self.attention_scale,
    )

    output = output.transpose(1, 2).contiguous()
    output = output.view(batch, length, width)

    return self.resid_dropout(self.out_proj(output))


def patch_attention(model):
    for block in model.blocks:
        block.attn.forward = MethodType(
            pcr_attention_forward,
            block.attn,
        )


patch_attention(phase1_model)

In [12]:
class Model1PCR(nn.Module):
    def __init__(
        self,
        pretrained_model,
        projection_dim,
        unfreeze_last_n,
    ):
        super().__init__()

        self.cfg = pretrained_model.cfg
        self.tok_emb = pretrained_model.tok_emb
        self.blocks = pretrained_model.blocks
        self.ln_f = pretrained_model.ln_f

        self.first_trainable = (
            len(self.blocks) - unfreeze_last_n
        )

        # Freeze the pretrained model first.
        self.requires_grad_(False)

        # Train only the last selected blocks.
        for block in self.blocks[self.first_trainable:]:
            block.requires_grad_(True)

        self.ln_f.requires_grad_(True)

        # Newly created projection is trainable.
        self.projection = nn.Linear(
            self.cfg.d_model,
            projection_dim,
            bias=False,
        )

    def train(self, mode=True):
        super().train(mode)

        # Frozen layers keep dropout disabled.
        for block in self.blocks[:self.first_trainable]:
            block.eval()

        return self

    def forward(self, input_ids, lengths):
        x = self.tok_emb(input_ids)

        for index, block in enumerate(self.blocks):
            if self.training and index >= self.first_trainable:
                x = activation_checkpoint(
                    block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x)

        # Right padding follows EOS, so select the last real token.
        rows = torch.arange(x.size(0), device=x.device)

        return self.ln_f(x[rows, lengths - 1])


pcr_model = Model1PCR(
    phase1_model,
    PROJECTION_DIM,
    UNFREEZE_LAST_N,
).to(device)

del phase1_model

print(
    'Total parameters:',
    sum(p.numel() for p in pcr_model.parameters()),
)

print(
    'Trainable parameters:',
    sum(
        p.numel()
        for p in pcr_model.parameters()
        if p.requires_grad
    ),
)

Total parameters: 56563360
Trainable parameters: 7680960


In [13]:
def encode_documents(model, documents):
    chunks = []
    owners = []

    for owner, document in enumerate(documents):
        chunks.extend(document)
        owners.extend([owner] * len(document))

    vectors = []

    for start in range(0, len(chunks), CHUNK_BATCH_SIZE):
        part = chunks[start:start + CHUNK_BATCH_SIZE]

        lengths = torch.tensor(
            [len(chunk) for chunk in part],
            device=device,
        )

        inputs = pad_sequence(
            [
                torch.tensor(chunk, dtype=torch.long)
                for chunk in part
            ],
            batch_first=True,
            padding_value=PAD_ID,
        ).to(device)

        vectors.append(model(inputs, lengths))

    vectors = torch.cat(vectors).float()
    owner_ids = torch.tensor(owners, device=device)

    # Add each chunk vector to its corresponding document.
    document_vectors = vectors.new_zeros(
        len(documents),
        model.cfg.d_model,
    )

    document_vectors = document_vectors.index_add(
        0,
        owner_ids,
        vectors,
    )

    counts = torch.bincount(
        owner_ids,
        minlength=len(documents),
    ).unsqueeze(1)

    document_vectors = document_vectors / counts
    document_vectors = model.projection(document_vectors)

    return F.normalize(document_vectors.float(), dim=-1)

In [14]:
train_candidate_ids = list(train_candidates)

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=60000,
    dtype=np.float32,
)

candidate_matrix = vectorizer.fit_transform(
    [
        train_candidates[case_id]['text']
        for case_id in train_candidate_ids
    ]
)

hard_negative_ids = {}

for query_id, row in train_queries.items():
    query_vector = vectorizer.transform([row['text']])

    similarities = (
        query_vector @ candidate_matrix.T
    ).toarray().ravel()

    ranked = np.argsort(-similarities, kind='stable')

    negatives = []

    for index in ranked:
        case_id = train_candidate_ids[index]

        if case_id not in row['relevant_candidates']:
            negatives.append(case_id)

        if len(negatives) == 20:
            break

    hard_negative_ids[query_id] = negatives

del vectorizer, candidate_matrix

training_ids = [
    query_id
    for query_id, row in train_queries.items()
    if row['relevant_candidates']
]

assert training_ids, (
    'Training requires queries with positive candidates.'
)

assert all(
    len(train_queries[q]['relevant_candidates'])
    < len(train_candidate_ids)
    for q in training_ids
)

train_loader = DataLoader(
    training_ids,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)

print('Training batches:', len(train_loader))

Training batches: 103


In [15]:
def sample_candidates(query_ids):
    selected = []

    for query_id in query_ids:
        positives = train_queries[query_id]['relevant_candidates']

        # Sample one positive for this query.
        selected.extend(random.sample(sorted(positives), min(2, len(positives))))

        hard = hard_negative_ids[query_id]

        selected.extend(
            random.sample(
                hard,
                min(HARD_NEGATIVES, len(hard)),
            )
        )

        other = [
            case_id
            for case_id in train_candidate_ids
            if case_id not in positives
        ]

        selected.extend(
            random.sample(
                other,
                min(RANDOM_NEGATIVES, len(other)),
            )
        )

    # A candidate appearing more than once is encoded once.
    return list(dict.fromkeys(selected))


def retrieval_loss(model, query_ids):
    candidate_ids = sample_candidates(query_ids)

    query_vectors = encode_documents(
        model,
        [train_query_chunks[q] for q in query_ids],
    )

    candidate_vectors = encode_documents(
        model,
        [train_candidate_chunks[c] for c in candidate_ids],
    )

    scores = (
        query_vectors @ candidate_vectors.T
    ) / TEMPERATURE

    positive_mask = torch.tensor(
        [
            [
                candidate_id
                in train_queries[query_id]['relevant_candidates']
                for candidate_id in candidate_ids
            ]
            for query_id in query_ids
        ],
        dtype=torch.bool,
        device=device,
    )

    # Every known positive in the batch receives positive supervision.
    log_probs = F.log_softmax(scores.float(), dim=-1)

    loss_per_query = -(
        log_probs * positive_mask
    ).sum(dim=1) / positive_mask.sum(dim=1)

    return loss_per_query.mean()

In [16]:
@torch.inference_mode()
def build_index(model, records):
    model.eval()

    ids = list(records)
    vectors = []

    for start in range(0, len(ids), INDEX_BATCH_SIZE):
        batch_ids = ids[start:start + INDEX_BATCH_SIZE]

        with amp_context():
            batch_vectors = encode_documents(
                model,
                [records[case_id] for case_id in batch_ids],
            )

        # Keep the complete candidate index on CPU.
        vectors.append(batch_vectors.cpu())

    return ids, torch.cat(vectors)


@torch.inference_mode()
def retrieve(model, query_chunks, candidate_chunks):
    model.eval()

    candidate_ids, candidate_vectors = build_index(
        model,
        candidate_chunks,
    )

    if len(candidate_ids) < 7:
        raise ValueError(
            'The candidate pool must contain at least seven cases.'
        )

    predictions = {}
    query_ids = list(query_chunks)

    for start in range(0, len(query_ids), INDEX_BATCH_SIZE):
        batch_ids = query_ids[start:start + INDEX_BATCH_SIZE]

        with amp_context():
            query_vectors = encode_documents(
                model,
                [query_chunks[q] for q in batch_ids],
            )

        scores = query_vectors.cpu() @ candidate_vectors.T
        top_indices = scores.topk(7, dim=1).indices.tolist()

        for query_id, indices in zip(batch_ids, top_indices):
            predictions[query_id] = [
                candidate_ids[index]
                for index in indices
            ]

    return predictions


def retrieval_scores(predictions, queries):
    assert set(predictions) == set(queries)

    hits = 0
    predicted_total = 0
    gold_total = 0

    for query_id, row in queries.items():
        ranked = predictions[query_id]

        assert len(ranked) == 7
        assert len(set(ranked)) == 7

        gold = row['relevant_candidates']

        hits += len(set(ranked) & gold)
        predicted_total += len(ranked)
        gold_total += len(gold)

    precision = hits / max(predicted_total, 1)
    recall = hits / max(gold_total, 1)

    f1 = (
        2 * hits
        / max(predicted_total + gold_total, 1)
    )

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


def evaluate_pcr(model):
    predictions = retrieve(
        model,
        dev_query_chunks,
        dev_candidate_chunks,
    )

    return retrieval_scores(predictions, dev_queries)

In [17]:
run_config = {
    'method': 'partial_finetuning',
    'max_length': MAX_LENGTH,
    'max_chunks': MAX_CHUNKS,
    'chunk_overlap': CHUNK_OVERLAP,
    'projection_dim': PROJECTION_DIM,
    'unfreeze_last_n': UNFREEZE_LAST_N,
    'pooling': 'chunk_eos_then_document_mean',
    'temperature': TEMPERATURE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'seed': SEED,
}


def train_pcr(model, train_loader):
    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        parameters,
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=use_amp and amp_dtype == torch.float16,
    )

    best_f1 = -1.0
    bad_epochs = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        total_loss = 0.0
        total_queries = 0

        for step, query_ids in enumerate(train_loader, 1):
            optimizer.zero_grad(set_to_none=True)

            with amp_context():
                loss = retrieval_loss(model, query_ids)

            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite retrieval loss.')

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                parameters,
                1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * len(query_ids)
            total_queries += len(query_ids)

            if step % 100 == 0 or step == len(train_loader):
                print(
                    f'Epoch {epoch} | '
                    f'Batch {step}/{len(train_loader)} | '
                    f'Loss: {loss.item():.4f}',
                    flush=True,
                )

        scores = evaluate_pcr(model)

        record = {
            'epoch': epoch,
            'train_loss': total_loss / total_queries,
            **scores,
        }

        history.append(record)
        print(record)

        (OUTPUT_DIR / 'history.json').write_text(
            json.dumps(history, indent=2),
            encoding='utf-8',
        )

        if scores['f1'] > best_f1:
            best_f1 = scores['f1']
            bad_epochs = 0

            torch.save(
                {
                    'model_state_dict': {
                        name: value.detach().cpu().clone()
                        for name, value
                        in model.state_dict().items()
                    },
                    'model_config': asdict(model.cfg),
                    'run_config': run_config,
                    'epoch': epoch,
                    'dev_scores': scores,
                },
                OUTPUT_DIR / 'best_f1.pt',
            )

            print('Saved best checkpoint.')

        else:
            bad_epochs += 1

            if bad_epochs >= PATIENCE:
                print('Early stopping.')
                break

    return history


history = train_pcr(pcr_model, train_loader)

shutil.copy2(
    MODEL_DIR / 'tokenizer.json',
    OUTPUT_DIR / 'tokenizer.json',
)

Epoch 1 | Batch 100/103 | Loss: 2.6972
Epoch 1 | Batch 103/103 | Loss: 1.5913
{'epoch': 1, 'train_loss': 2.970040300047091, 'precision': 0.25786924939467315, 'recall': 0.21891058581706063, 'f1': 0.2367982212340189}
Saved best checkpoint.
Epoch 2 | Batch 100/103 | Loss: 2.6875
Epoch 2 | Batch 103/103 | Loss: 0.9928
{'epoch': 2, 'train_loss': 2.474966487510989, 'precision': 0.2917675544794189, 'recall': 0.24768756423432683, 'f1': 0.2679266259032796}
Saved best checkpoint.
Epoch 3 | Batch 100/103 | Loss: 2.1097
Epoch 3 | Batch 103/103 | Loss: 1.5891
{'epoch': 3, 'train_loss': 2.3542594612009524, 'precision': 0.31113801452784506, 'recall': 0.2641315519013361, 'f1': 0.2857142857142857}
Saved best checkpoint.
Epoch 4 | Batch 100/103 | Loss: 2.0699
Epoch 4 | Batch 103/103 | Loss: 5.5616
{'epoch': 4, 'train_loss': 2.35622117802446, 'precision': 0.3171912832929782, 'recall': 0.2692702980472765, 'f1': 0.29127292940522514}
Saved best checkpoint.
Epoch 5 | Batch 100/103 | Loss: 2.3025
Epoch 5 | Ba

PosixPath('/kaggle/working/phase7_model1_partial/tokenizer.json')

In [18]:
saved = torch.load(
    OUTPUT_DIR / 'best_f1.pt',
    map_location='cpu',
    weights_only=True,
)

pcr_model.load_state_dict(
    saved['model_state_dict'],
    strict=True,
)

pcr_model.eval()

print('Best epoch:', saved['epoch'])
print('Dev scores:', saved['dev_scores'])

del saved

test_query_chunks = prepare_cases(test_queries)
test_candidate_chunks = prepare_cases(test_candidates)

test_predictions = retrieve(
    pcr_model,
    test_query_chunks,
    test_candidate_chunks,
)

Best epoch: 7
Dev scores: {'precision': 0.3280871670702179, 'recall': 0.27852004110996914, 'f1': 0.30127848804891605}


In [19]:
assert set(test_predictions) == set(test_queries)

candidate_pool = set(test_candidates)

for ranked in test_predictions.values():
    assert len(ranked) == 7
    assert len(set(ranked)) == 7
    assert set(ranked) <= candidate_pool

prediction_path = OUTPUT_DIR / 'prediction.json'

prediction_path.write_text(
    json.dumps(test_predictions, indent=2),
    encoding='utf-8',
)

with zipfile.ZipFile(
    OUTPUT_DIR / 'submission.zip',
    'w',
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname='prediction.json',
    )

print('Submission:', OUTPUT_DIR / 'submission.zip')

Submission: /kaggle/working/phase7_model1_partial/submission.zip


In [20]:
HYBRID_DIR = OUTPUT_DIR / 'hybrid'
HYBRID_DIR.mkdir(parents=True, exist_ok=True)

NEURAL_CHECKPOINT = OUTPUT_DIR / 'best_f1.pt'

RETRIEVE_TOP_K = 100
RRF_CONSTANT = 60

# 1.0 keeps the neural-only baseline as an option.
NEURAL_WEIGHTS = [1.0, 0.75, 0.5, 0.25]

saved = torch.load(
    NEURAL_CHECKPOINT,
    map_location='cpu',
    weights_only=True,
)

pcr_model.load_state_dict(
    saved['model_state_dict'],
    strict=True,
)

pcr_model.eval()

print('Loaded neural model from epoch:', saved['epoch'])
del saved

Loaded neural model from epoch: 7


In [21]:
@torch.inference_mode()
def neural_rankings(model, query_chunks, candidate_chunks):
    model.eval()

    candidate_ids, candidate_vectors = build_index(
        model,
        candidate_chunks,
    )

    if len(candidate_ids) < 7:
        raise ValueError('At least seven candidates are required.')

    count = min(RETRIEVE_TOP_K, len(candidate_ids))
    query_ids = list(query_chunks)
    rankings = {}

    for start in range(0, len(query_ids), INDEX_BATCH_SIZE):
        batch_ids = query_ids[start:start + INDEX_BATCH_SIZE]

        with amp_context():
            vectors = encode_documents(
                model,
                [query_chunks[q] for q in batch_ids],
            )

        scores = vectors.cpu() @ candidate_vectors.T

        indices = scores.topk(
            count,
            dim=1,
        ).indices.tolist()

        for query_id, row in zip(batch_ids, indices):
            rankings[query_id] = [
                candidate_ids[i] for i in row
            ]

    return rankings

In [22]:
def text_rankings(queries, candidates):
    candidate_ids = list(candidates)

    if len(candidate_ids) < 7:
        raise ValueError('At least seven candidates are required.')

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_features=60000,
        dtype=np.float32,
    )

    matrix = vectorizer.fit_transform(
        [
            candidates[case_id]['text']
            for case_id in candidate_ids
        ]
    )

    rankings = {}
    count = min(RETRIEVE_TOP_K, len(candidate_ids))

    for query_id, row in queries.items():
        query_vector = vectorizer.transform([row['text']])

        scores = (
            query_vector @ matrix.T
        ).toarray().ravel()

        indices = np.argsort(
            -scores,
            kind='stable',
        )[:count]

        rankings[query_id] = [
            candidate_ids[i] for i in indices
        ]

    return rankings

In [23]:
def fuse_rankings(neural, lexical, neural_weight):
    assert set(neural) == set(lexical)

    predictions = {}

    for query_id in neural:
        scores = {}

        sources = [
            (neural[query_id], neural_weight),
            (lexical[query_id], 1.0 - neural_weight),
        ]

        for ranking, weight in sources:
            if weight == 0:
                continue

            for rank, candidate_id in enumerate(
                ranking,
                start=1,
            ):
                contribution = weight / (RRF_CONSTANT + rank)

                scores[candidate_id] = (
                    scores.get(candidate_id, 0.0)
                    + contribution
                )

        # Candidate ID resolves exact score ties consistently.
        ranked = sorted(
            scores,
            key=lambda candidate_id: (
                -scores[candidate_id],
                candidate_id,
            ),
        )

        predictions[query_id] = ranked[:7]

    return predictions

In [24]:
dev_neural = neural_rankings(
    pcr_model,
    dev_query_chunks,
    dev_candidate_chunks,
)

dev_text = text_rankings(
    dev_queries,
    dev_candidates,
)

# Diagnostic text-only baseline.
text_only = {
    query_id: ranked[:7]
    for query_id, ranked in dev_text.items()
}

print(
    'Text-only baseline:',
    retrieval_scores(text_only, dev_queries),
)

Text-only baseline: {'precision': 0.3789346246973366, 'recall': 0.32168550873586843, 'f1': 0.3479710950528071}


In [25]:
results = []

for weight in NEURAL_WEIGHTS:
    predictions = fuse_rankings(
        dev_neural,
        dev_text,
        neural_weight=weight,
    )

    scores = retrieval_scores(
        predictions,
        dev_queries,
    )

    result = {
        'neural_weight': weight,
        'text_weight': 1.0 - weight,
        **scores,
    }

    results.append(result)
    print(result)

best_result = max(
    results,
    key=lambda row: row['f1'],
)

BEST_WEIGHT = best_result['neural_weight']

print('Selected on Dev:', best_result)

(HYBRID_DIR / 'dev_results.json').write_text(
    json.dumps(results, indent=2),
    encoding='utf-8',
)

(HYBRID_DIR / 'settings.json').write_text(
    json.dumps(
        {
            'neural_checkpoint': str(NEURAL_CHECKPOINT),
            'neural_weight': BEST_WEIGHT,
            'retrieve_top_k': RETRIEVE_TOP_K,
            'rrf_constant': RRF_CONSTANT,
        },
        indent=2,
    ),
    encoding='utf-8',
)

{'neural_weight': 1.0, 'text_weight': 0.0, 'precision': 0.3280871670702179, 'recall': 0.27852004110996914, 'f1': 0.30127848804891605}
{'neural_weight': 0.75, 'text_weight': 0.25, 'precision': 0.3583535108958838, 'recall': 0.30421377183967113, 'f1': 0.3290717065036131}
{'neural_weight': 0.5, 'text_weight': 0.5, 'precision': 0.3765133171912833, 'recall': 0.31963001027749227, 'f1': 0.34574763757643134}
{'neural_weight': 0.25, 'text_weight': 0.75, 'precision': 0.3801452784503632, 'recall': 0.3227132579650565, 'f1': 0.349082823790995}
Selected on Dev: {'neural_weight': 0.25, 'text_weight': 0.75, 'precision': 0.3801452784503632, 'recall': 0.3227132579650565, 'f1': 0.349082823790995}


149

In [26]:
test_query_chunks = prepare_cases(test_queries)
test_candidate_chunks = prepare_cases(test_candidates)

test_neural = neural_rankings(
    pcr_model,
    test_query_chunks,
    test_candidate_chunks,
)

test_text = text_rankings(
    test_queries,
    test_candidates,
)

hybrid_predictions = fuse_rankings(
    test_neural,
    test_text,
    neural_weight=BEST_WEIGHT,
)

In [27]:
assert set(hybrid_predictions) == set(test_queries)

candidate_pool = set(test_candidates)

for ranked in hybrid_predictions.values():
    assert len(ranked) == 7
    assert len(set(ranked)) == 7
    assert set(ranked) <= candidate_pool

prediction_path = HYBRID_DIR / 'prediction.json'

prediction_path.write_text(
    json.dumps(hybrid_predictions, indent=2),
    encoding='utf-8',
)

with zipfile.ZipFile(
    HYBRID_DIR / 'submission.zip',
    'w',
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname='prediction.json',
    )

print('Submit:', HYBRID_DIR / 'submission.zip')

Submit: /kaggle/working/phase7_model1_partial/hybrid/submission.zip


In [28]:
current_dev_predictions = fuse_rankings(
    dev_neural,
    dev_text,
    BEST_WEIGHT,
)

current_scores = retrieval_scores(
    current_dev_predictions,
    dev_queries,
)

gold_total = 0
found_in_shortlist = 0
maximum_top7_hits = 0

for query_id, row in dev_queries.items():
    gold = set(row['relevant_candidates'])

    # Union of neural and text retrieval lists.
    shortlist = (
        set(dev_neural[query_id])
        | set(dev_text[query_id])
    )

    available = len(gold & shortlist)

    gold_total += len(gold)
    found_in_shortlist += available

    # Even a perfect reranker can return only seven cases.
    maximum_top7_hits += min(7, available)

shortlist_recall = (
    found_in_shortlist / max(gold_total, 1)
)

oracle_f1 = (
    2 * maximum_top7_hits
    / max(7 * len(dev_queries) + gold_total, 1)
)

print('Selected neural weight:', BEST_WEIGHT)
print('Current Dev F1@7:', round(current_scores['f1'], 4))
print('Shortlist recall:', round(shortlist_recall, 4))
print('Best possible F1@7 within shortlist:', round(oracle_f1, 4))

Selected neural weight: 0.25
Current Dev F1@7: 0.3491
Shortlist recall: 0.7924
Best possible F1@7 within shortlist: 0.5737
